# Multi-Node Training on SageMaker Training job

In [1]:
# ## Update sagemaker python sdk version
!pip install -U sagemaker

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


## Set model, Code and data

In [1]:
import sagemaker
from sagemaker import get_execution_role

sess = sagemaker.Session()
role = get_execution_role()
sagemaker_default_bucket = sess.default_bucket()
region = sess.boto_session.region_name
print("sagemaker_default_bucket:", sagemaker_default_bucket)
print("sagemaker_region:", region)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
sagemaker_default_bucket: sagemaker-us-east-1-596899493901
sagemaker_region: us-east-1


## upload pretrain models to s3

In [2]:
!pip install huggingface_hub

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:
# Code language: python
from huggingface_hub import snapshot_download
from pathlib import Path

local_cache_path = Path("../deepseek_coder")
local_cache_path.mkdir(exist_ok=True)

model_name = "deepseek-ai/deepseek-coder-6.7b-base"

# Only download pytorch checkpoint files
allow_patterns = ["*"]

model_download_path = snapshot_download(
    repo_id=model_name,
    cache_dir=local_cache_path,
    allow_patterns=allow_patterns,
)
model_snapshot_path = list(local_cache_path.glob("**/snapshots/*"))[0]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

In [9]:
sagemaker_default_bucket

NameError: name 'sagemaker_default_bucket' is not defined

In [3]:
!aws s3 cp {model_snapshot_path} s3://{sagemaker_default_bucket}/Foundation-Models/deepseek_coder --recursive

upload: deepseek_coder/models--deepseek-ai--deepseek-coder-6.7b-base/snapshots/ce2207a8bfef3ee92bd7dd4cc31c52cfa0046912/config.json to s3://sagemaker-us-east-1-596899493901/Foundation-Models/deepseek_coder/config.json
upload: deepseek_coder/models--deepseek-ai--deepseek-coder-6.7b-base/snapshots/ce2207a8bfef3ee92bd7dd4cc31c52cfa0046912/generation_config.json to s3://sagemaker-us-east-1-596899493901/Foundation-Models/deepseek_coder/generation_config.json
upload: deepseek_coder/models--deepseek-ai--deepseek-coder-6.7b-base/snapshots/ce2207a8bfef3ee92bd7dd4cc31c52cfa0046912/LICENSE to s3://sagemaker-us-east-1-596899493901/Foundation-Models/deepseek_coder/LICENSE
upload: deepseek_coder/models--deepseek-ai--deepseek-coder-6.7b-base/snapshots/ce2207a8bfef3ee92bd7dd4cc31c52cfa0046912/.gitattributes to s3://sagemaker-us-east-1-596899493901/Foundation-Models/deepseek_coder/.gitattributes
upload: deepseek_coder/models--deepseek-ai--deepseek-coder-6.7b-base/snapshots/ce2207a8bfef3ee92bd7dd4cc31c5

## Setup for wandb

In [4]:
!pip install wandb

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [3]:
import wandb
wandb.login()

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: 407383787 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Submit Training job

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.pytorch import PyTorch
from datetime import datetime


instance_count = 2
# instance_type = 'ml.p4d.24xlarge'
instance_type = 'ml.g5.48xlarge'  ## 8*24G
max_time = 200000

# Get the current time
current_time = datetime.now()

wandb.sagemaker_auth(path="./")
# Format the current time as a string
formatted_time = current_time.strftime("%Y%m%d%H%M%S")
print(formatted_time)

base_job_name = 'deepseek6-7B-finetune'
environment = {
    'NODE_NUMBER':str(instance_count),
    'MODEL_S3_PATH': f's3://sagemaker-us-east-1-596899493901/Foundation-Models/deepseek_coder', # source model files
    'MODEL_LOCAL_PATH': '/tmp/pretrain_model',
    'OUTPUT_MODEL_S3_PATH': f's3://{sagemaker_default_bucket}/deepseek-coder-6.7b-base/finetuned_model/', # destination
}

estimator = PyTorch(entry_point='entry.py',
                            source_dir='./',
                            role=role,
                            base_job_name=base_job_name,
                            environment=environment,
                            framework_version='2.1.0',
                            py_version='py310',
                            script_mode=True,
                            instance_count=instance_count,
                            instance_type=instance_type,
                            max_run=max_time)

# # data in channel will be automatically copied to each node - /opt/ml/input/data/train1
#input_channel = {'train': f's3://{sagemaker_default_bucket}/datasets/qiandao/{version}/train.json'}
estimator.fit()

20250820144233


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: deepseek6-7B-finetune-2025-08-20-14-42-33-527


2025-08-20 14:42:42 Starting - Starting the training job
2025-08-20 14:42:42 Pending - Training job waiting for capacity.................................
2025-08-20 14:48:07 Pending - Preparing the instances for training.........
2025-08-20 14:49:31 Downloading - Downloading input data...
2025-08-20 14:49:51 Downloading - Downloading the training image...............
2025-08-20 14:52:23 Training - Training image download completed. Training in progress.....bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.a